In [1]:
from pebble import ProcessPool
from src.tools.dataloaders import load_problem_subset, load_ground_truth_solutions, load_test_cases, get_data_dir, load_working_solutions
from src.code_evaluation.executor_standard import execute_code, check
from core.utils import setup_logging
from tqdm.auto import tqdm
import logging
setup_logging('info')

logger = logging.getLogger(__name__)

already_processed = load_working_solutions()
problems = load_problem_subset("INTERVIEW", require_solutions=True)(get_data_dir() / "test", num_problems=5000)
problems = {k: v for k, v in problems.items() if k not in already_processed}
logger.info(f"Processing {len(problems)} problems")
all_solutions = load_ground_truth_solutions(problems.keys())
test_cases = load_test_cases(problems.keys())

working_solutions = {}
all_results = {}
with ProcessPool(16) as p:
    futures = {}
    first_passing_solution = None
    for problem_id, solutions in tqdm(all_solutions.items()):
        all_results[problem_id] = {}
        logger.debug(f"Evaluating {len(solutions)} solutions for problem {problem_id}")
        for i, solution in enumerate(solutions):
            if first_passing_solution:
                break
            results = []
            futures = []
            for test_case in test_cases[problem_id]:
                futures.append(p.schedule(execute_code, (solution, test_case['input']), timeout=5))
            
            for future in futures:
                try:
                    result = future.result(timeout=5)
                    results.append(result)
                except Exception as e:
                    results.append("ERROR")
            
            all_passed = True
            all_results[problem_id][i] = []
            for result, test_case in zip(results, test_cases[problem_id]):
                # print(f"input: {repr(test_case['input'])}")
                # print(f"expected: {repr(test_case['output'])}")
                # print(f"actual: {repr(result)}")
                all_results[problem_id][i].append((test_case['input'], result, test_case['output']))
                if not check(test_case['input'], result, test_case['output']):
                    all_passed = False
                    break
            
            if all_passed:
                first_passing_solution = solution
                break
        
        if first_passing_solution:
            logger.debug(f"Problem {problem_id}: Found a passing solution")
            working_solutions[problem_id] = first_passing_solution
            first_passing_solution = None
        else:
            logger.debug(f"Problem {problem_id}: No passing solution found")
            #print(all_results[problem_id])
            working_solutions[problem_id] = None

import json

# Assuming 'all_results' is the dictionary we want to save
output_file = get_data_dir() / 'working_solutions.json'

# Save the dictionary to a JSON file
merged_results = {**already_processed, **working_solutions}

with open(output_file, 'w') as f:
    json.dump(merged_results, f, indent=4)

print(f"Results saved to {output_file}")

#print(working_solutions)
#print(all_results)

2024-09-23 21:05:56 [INFO] (core.utils) Logging level set to info
2024-09-23 21:05:56 [INFO] (__main__) Processing 0 problems


0it [00:00, ?it/s]

Results saved to /home/caleb/workspace/control/data/APPS/working_solutions.json


In [2]:
import json

# Assuming 'all_results' is the dictionary we want to save
output_file = get_data_dir() / 'working_solutions.json'

# Save the dictionary to a JSON file

with open(output_file, 'w') as f:
    json.dump(working_solutions, f, indent=4)

print(f"Results saved to {output_file}")


Results saved to /home/caleb/workspace/control/data/APPS/working_solutions.json


In [4]:
import json

# Assuming 'all_results' is the dictionary we want to save
output_file = get_data_dir() / 'working_solutions.json'
output_file_old = get_data_dir() / 'working_solutions_old.json'

with open(output_file_old, 'r') as f:
    old_results = json.load(f)
with open(output_file, 'r') as f:
    new_results = json.load(f)

print(len(old_results))
print(len(new_results))


merged = {**old_results, **new_results}

# Save the dictionary to a JSON file

with open(output_file, 'w') as f:
    json.dump(merged, f, indent=4)

print(f"Results saved to {output_file}")


1500
1227
Results saved to /home/caleb/workspace/control/data/APPS/working_solutions.json
